In [5]:
from dask.diagnostics import ProgressBar
ProgressBar().register()

In [6]:
import shutil
from pathlib import Path

# Wipe the entire dataset cache so it rebuilds cleanly
#shutil.rmtree(Path("Downloads/images"))  # old prefixed images
#import os
#os.remove("Downloads/alzheimers-metadata.csv")  # old CSV with prefixed paths

import platformdirs, shutil
from pathlib import Path

cache_root = Path(platformdirs.user_cache_dir(appname="pyhealth"))
shutil.rmtree(cache_root)
print("Cache cleared")

Cache cleared


In [7]:
from datasets import load_dataset

dataset_to_load = "Falah/Alzheimer_MRI"

In [8]:
import torch

DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"

METRICS = ["accuracy", "f1_macro", "balanced_accuracy"]

In [9]:
"""3D CNN model for Alzheimer's disease detection from structural MRI.

Reference:
    Liu et al. "On the design of convolutional neural networks for automatic
    detection of Alzheimer's disease." ML4H @ NeurIPS 2019.
    https://arxiv.org/abs/1911.03740

    Liu et al. "Generalizable deep learning model for early Alzheimer's
    disease detection from structural MRIs." Scientific Reports, 2022.
    https://www.nature.com/articles/s41598-022-20674-x
"""

from typing import Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from pyhealth.datasets import SampleDataset
from pyhealth.models import BaseModel

class AlzheimerCNN(BaseModel):
    def __init__(
        self,
        dataset: SampleDataset,
        init_channels: int = 32,
        classifier_hidden_dim: int = 128,
        dropout: float = 0.5,
        **kwargs,
    ) -> None:
        super(AlzheimerCNN, self).__init__(dataset=dataset)

        self.init_channels = init_channels
        self.classifier_hidden_dim = classifier_hidden_dim

        # ── Block 1: Conv2d(1 → C) → InstanceNorm → LeakyReLU → MaxPool ──
        self.block1 = nn.Sequential(
            nn.Conv2d(1, init_channels, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 2: Conv2d(C → 2C) → InstanceNorm → LeakyReLU → MaxPool ─
        self.block2 = nn.Sequential(
            nn.Conv2d(init_channels, init_channels * 2, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 3: Conv2d(2C → 4C) → InstanceNorm → LeakyReLU → MaxPool ─
        self.block3 = nn.Sequential(
            nn.Conv2d(init_channels * 2, init_channels * 4, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 4),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 4: Conv2d(4C → 8C) → InstanceNorm → LeakyReLU → AdaptiveAvgPool ─
        self.block4 = nn.Sequential(
            nn.Conv2d(init_channels * 4, init_channels * 8, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 8),
            nn.LeakyReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # ── Classifier: FC(8C → hidden_dim) → LeakyReLU → Dropout → FC(hidden_dim → output) ─
        output_size = self.get_output_size()
        self.classifier = nn.Sequential(
            nn.Linear(init_channels * 8, classifier_hidden_dim),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(classifier_hidden_dim, output_size),
        )
    def forward(self, **kwargs) -> Dict[str, torch.Tensor]:
        # ── Extract inputs; self.device handles mps/cuda/cpu automatically ─
        x = kwargs[self.feature_keys[0]].to(self.device)
        labels = kwargs[self.label_keys[0]].to(self.device)

        # ── Convolutional backbone ────────────────────────────────────────
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        # ── Flatten: (B, 8C, 1, 1) → (B, 8C) ────────────────────────────
        emb = torch.flatten(x, 1)

        # ── Classifier head ───────────────────────────────────────────────
        logits = self.classifier(emb)

        # ── Loss + metrics via BaseModel helpers ──────────────────────────
        loss = self.get_loss_function()(logits, labels)
        y_prob = self.prepare_y_prob(logits)

        return {
            "loss": loss,
            "y_prob": y_prob,
            "y_true": labels,
            "logit": logits,
        }

Create Task in PyHealth

In [10]:
from typing import Any, Dict, List

from pyhealth.tasks.base_task import BaseTask


class AlzheimersClassification(BaseTask):
    """Task for classifying Alzheimer's disease from brain images."""

    task_name: str = "AlzheimersClassification"

    # Input: image path
    input_schema: Dict = {
        "image": ("image", {"mode": "L", "image_size": 128})
    }

    # Output: classification label
    output_schema: Dict[str, str] = {
        "label": "multiclass"
    }

    def __call__(self, patient: Any) -> List[Dict[str, Any]]:
        """Convert patient events into model-ready samples."""

        # Must match your table name in BaseDataset
        events = patient.get_events(event_type="alzheimers")

        # For image datasets, usually 1 image per "patient"
        assert len(events) == 1, f"Expected 1 image, got {len(events)}"

        event = events[0]

        image = event.path   # comes from metadata CSV
        label = event.label  # comes from metadata CSV

        return [{
            "image": image,
            "label": label
            # age removed — not present in Falah/Alzheimer_MRI
        }]

# Create Sample Dataset in PyHealth

In [11]:
import logging
import os
from pathlib import Path
from typing import Optional
import yaml

import pandas as pd
from datasets import load_dataset

from pyhealth.datasets.base_dataset import BaseDataset

logger = logging.getLogger(__name__)


class AlzheimersDataset(BaseDataset):
    """PyHealth-compatible Alzheimer's dataset (auto-download + process)."""

    def __init__(
        self,
        root: str,
        dataset_name: Optional[str] = None,
        config_path: Optional[str] = None,
        cache_dir: Optional[str] = None,
        num_workers: int = 1,
        dev: bool = False,
        hf_dataset_name: str = dataset_to_load,  # configurable based on required dataset
    ) -> None:

        self.hf_dataset_name = hf_dataset_name

        try:
            base_path = Path(__file__).parent
        except NameError:
            base_path = Path.cwd()  # notebook working directory
        
        if config_path is None:
            config_path = base_path / "configs" / "alzheimers.yaml"

        metadata_path = os.path.join(root, "alzheimers-metadata.csv")

        if not os.path.exists(metadata_path):
            logger.info(f"Preparing dataset from {hf_dataset_name}...")
            self.prepare_metadata(root)

        super().__init__(
            root=root,
            tables=["alzheimers"],
            dataset_name=dataset_name or "alzheimers",
            config_path=config_path,
            cache_dir=cache_dir,
            num_workers=num_workers,
            dev=dev,
        )

    def prepare_metadata(self, root: str) -> None:
        """Download + convert HF dataset into PyHealth format."""
        os.makedirs(root, exist_ok=True)
        image_dir = os.path.join(root, "images")
        os.makedirs(image_dir, exist_ok=True)
    
        metadata_path = os.path.join(root, "alzheimers-metadata.csv")
    
        # Load existing metadata for duplicate checking
        if os.path.exists(metadata_path):
            existing_df = pd.read_csv(metadata_path)
            existing_paths = set(existing_df["path"].tolist())
            next_idx = existing_df["patient_id"].astype(int).max() + 1
        else:
            existing_df = None
            existing_paths = set()
            next_idx = 0
    
        # Falah/Alzheimer_MRI only has a 'train' split, with columns: image, label
        # No age or other metadata available in this dataset
        # ds = load_dataset(self.hf_dataset_name)  # OLD: iterated over splits with split-aware patient_id
        ds = load_dataset(self.hf_dataset_name, split="train")
    
        records = []
        idx = next_idx
    
        # OLD: for split in ds:
        #          for i, item in enumerate(ds[split]):
        #              image = item["image"]
        #              label = item.get("label")
        #              if label is None: raise ValueError(...)
        #              img_path = os.path.join(image_dir, f"{split}_{i}.png")
        #              records.append({..., "age": item.get("age", None)})
        for i, item in enumerate(ds):
            img_path = os.path.join(image_dir, f"{i}.png")
    
            # Skip if already recorded in CSV
            if img_path in existing_paths:
                logger.debug(f"Skipping duplicate: {img_path}")
                continue
    
            image = item["image"]
            label = item.get("label")
            if label is None:
                raise ValueError(f"Missing label at index {i}")
    
            # Skip disk write if image already saved
            if not os.path.isfile(img_path):
                image.save(img_path)
                if not os.path.isfile(img_path):
                    raise FileNotFoundError(f"Image save failed: {img_path}")
            else:
                logger.debug(f"Image already on disk, skipping save: {img_path}")
    
            # No age field in Falah/Alzheimer_MRI — omitted
            # OLD: "age": item.get("age", None)
            records.append({
                "patient_id": str(idx),
                "path": img_path,
                "label": str(label),
            })
            idx += 1
    
        if len(records) == 0 and existing_df is None:
            raise ValueError("Dataset is empty")
    
        if records:
            new_df = pd.DataFrame(records)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True) if existing_df is not None else new_df
            combined_df.to_csv(metadata_path, index=False)
            logger.info(f"Added {len(records)} new records. Total: {len(combined_df)}.")
        else:
            logger.info("No new records to add, metadata is already up to date.")

    @property
    def default_task(self):
        from .alzheimers_task import AlzheimersClassification
        return AlzheimersClassification()

# Instantiate and use PyHealth Trainer

In [13]:
from pyhealth.datasets import split_by_sample, get_dataloader
from pyhealth.trainer import Trainer

dataset = AlzheimersDataset(root = "Downloads")
dataset_samples = dataset.set_task(AlzheimersClassification(), num_workers = 1)

train_samples, val_samples, test_samples = split_by_sample(dataset_samples, [.6, .2, .2])

# Init model
model = AlzheimerCNN(
    dataset=train_samples,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
)

#Num workers is 0 because I am running on MacOS, was previously 4
train_loader = get_dataloader(train_samples, batch_size=32, shuffle=True)
val_loader = get_dataloader(val_samples,   batch_size=32, shuffle=False)
test_loader = get_dataloader(test_samples,  batch_size=32, shuffle=False)

# Set up PyHealth Trainer
trainer = Trainer(
    model=model,
    metrics=METRICS,
    device=DEVICE
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 1e-4, "weight_decay": 1e-5}
)

Initializing alzheimers dataset from Downloads (dev mode: False)
No cache_dir provided. Using default cache dir: C:\Users\spicy\AppData\Local\pyhealth\pyhealth\Cache\5e3d2e44-3165-596e-89d3-ed772a0ed564
Setting task AlzheimersClassification for alzheimers base dataset...
Task cache paths: task_df=C:\Users\spicy\AppData\Local\pyhealth\pyhealth\Cache\5e3d2e44-3165-596e-89d3-ed772a0ed564\tasks\AlzheimersClassification_7e906453-ea6b-5cc3-8506-3e00427d79ba\task_df.ld, samples=C:\Users\spicy\AppData\Local\pyhealth\pyhealth\Cache\5e3d2e44-3165-596e-89d3-ed772a0ed564\tasks\AlzheimersClassification_7e906453-ea6b-5cc3-8506-3e00427d79ba\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
No cached event dataframe found. Creating: C:\Users\spicy\AppData\Local\pyhealth\pyhealth\Cache\5e3d2e44-3165-596e-89d3-ed772a0ed564\global_event_df.parquet
Scanning table: alzheimers from C:\Users\spicy\Downloads\598AI\FinalProject\Downloads\alzheimers-metadata

  0%|          | 0/5120 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████| 5120/5120 [00:01<00:00, 4411.94it/s]

Worker 0 finished processing patients.
Fitting processors on the dataset...


...

--- Eval epoch-48, step-4704 ---
accuracy: 0.6758
f1_macro: 0.4480
balanced_accuracy: 0.4470
loss: 0.7206



Epoch 49 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-49, step-4800 ---
loss: 0.5014


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 12.02it/s]

--- Eval epoch-49, step-4800 ---
accuracy: 0.6885
f1_macro: 0.4203
balanced_accuracy: 0.4289
loss: 0.6719


# Abilation number one per the proposal

In [17]:
class AlzheimerCNNNormVariant(BaseModel):
 
    def __init__(
        self,
        dataset,
        norm_type: str = "instance",
        num_groups: int = 8,
        init_channels: int = 32,
        classifier_hidden_dim: int = 128,
        dropout: float = 0.5,
        **kwargs,
    ) -> None:
        super(AlzheimerCNNNormVariant, self).__init__(dataset=dataset)

        if norm_type not in ("instance", "group", "layer"):
            raise ValueError(
                f"norm_type must be 'instance', 'group', or 'layer', got '{norm_type}'"
            )

        self.norm_type = norm_type
        self.num_groups = num_groups
        self.init_channels = init_channels

        def _get_norm(channels: int) -> nn.Module:
            """Build the appropriate normalization layer for a given channel count.

            Args:
                channels: Number of feature map channels to normalise.

            Returns:
                An nn.Module normalisation layer.
            """
            if norm_type == "instance":
                return nn.InstanceNorm2d(channels)
            elif norm_type == "group":
                # Ensure num_groups evenly divides channels
                groups = min(num_groups, channels)
                while channels % groups != 0:
                    groups //= 2
                return nn.GroupNorm(groups, channels)
            else:
                # layer norm: GroupNorm(1, C) is equivalent to
                # LayerNorm over the channel dimension
                return nn.GroupNorm(1, channels)

        # ── Block 1: Conv2d(1 → C) → Norm → LeakyReLU → MaxPool ──────────
        self.block1 = nn.Sequential(
            nn.Conv2d(1, init_channels, kernel_size=3, stride=1, padding=1),
            _get_norm(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 2: Conv2d(C → 2C) → Norm → LeakyReLU → MaxPool ─────────
        self.block2 = nn.Sequential(
            nn.Conv2d(init_channels, init_channels * 2, kernel_size=3, stride=1, padding=1),
            _get_norm(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 3: Conv2d(2C → 4C) → Norm → LeakyReLU → MaxPool ────────
        self.block3 = nn.Sequential(
            nn.Conv2d(init_channels * 2, init_channels * 4, kernel_size=3, stride=1, padding=1),
            _get_norm(init_channels * 4),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ── Block 4: Conv2d(4C → 8C) → Norm → LeakyReLU → AdaptiveAvgPool ─
        self.block4 = nn.Sequential(
            nn.Conv2d(init_channels * 4, init_channels * 8, kernel_size=3, stride=1, padding=1),
            _get_norm(init_channels * 8),
            nn.LeakyReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # ── Classifier: FC(8C → hidden) → LeakyReLU → Dropout → FC(hidden → output) ─
        output_size = self.get_output_size()
        self.classifier = nn.Sequential(
            nn.Linear(init_channels * 8, classifier_hidden_dim),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(classifier_hidden_dim, output_size),
        )

    def forward(self, **kwargs) -> Dict[str, torch.Tensor]:
        # Input and Device Management
        x = kwargs[self.feature_keys[0]].to(self.device)
        labels = kwargs[self.label_keys[0]].to(self.device)

        # Convolutions
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        # Flatten
        emb = torch.flatten(x, 1)

        logits = self.classifier(emb)

        # Metrics
        loss = self.get_loss_function()(logits, labels)
        y_prob = self.prepare_y_prob(logits)

        return {
            "loss": loss,
            "y_prob": y_prob,
            "y_true": labels,
            "logit": logits,
        }

In [18]:
from torch.utils.data import DataLoader

# Instantiate the Model
model = AlzheimerCNNNormVariant(
    dataset=train_samples,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
    norm_type='group',
)

#Num workers is 0 because I am running on MacOS, was previously 4
train_loader = get_dataloader(train_samples, batch_size=32, shuffle=True)
val_loader = get_dataloader(val_samples,   batch_size=32, shuffle=False)
test_loader = get_dataloader(test_samples,  batch_size=32, shuffle=False)

# Set up PyHealth Trainer
trainer = Trainer(
    model=model,
    metrics=METRICS,
    device=DEVICE
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 1e-4, "weight_decay": 1e-5}
)

AlzheimerCNNNormVariant(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 32, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))


Epoch 0 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-0, step-96 ---
loss: 1.1009


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 15.55it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.5098
f1_macro: 0.1688
balanced_accuracy: 0.2500
loss: 1.0397



...

--- Train epoch-5, step-576 ---
loss: 0.9520


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 15.61it/s]

--- Eval epoch-5, step-576 ---
accuracy: 0.5420
f1_macro: 0.2884
balanced_accuracy: 0.3158
loss: 0.9316



Epoch 6 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Abilation Number 2

In [9]:
class DepthwiseSeparableConv3d(nn.Module):
    """
    3D Depthwise Separable Convolution block.
    Replaces a standard Conv3d(in_ch, out_ch, k) with:
      1) Depthwise:  Conv3d(in_ch, in_ch, k, groups=in_ch)  — spatial filtering per channel
      2) Pointwise: Conv3d(in_ch, out_ch, 1)               — channel mixing
    Parameter reduction: ~(k^3) x fewer params vs standard Conv3d.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(
            in_channels, in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels   # one filter per input channel
        )
        self.pointwise = nn.Conv3d(
            in_channels, out_channels,
            kernel_size=1        # 1x1x1 — mixes channels without touching spatial dims
        )

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x


class PyHealthAlzheimersCNN_DepthwiseSep3D(nn.Module):
    """
    Extension 2: 3D Depthwise Separable Convolutions.
    The 2D MRI slices are unsqueezed to (B, 1, 1, H, W) — a 3D volume with depth=1.
    Each block uses DepthwiseSeparableConv3d instead of Conv2d, preserving the
    model's ability to reason about 3D structure while cutting parameter count ~9x
    per layer (for kernel_size=3).
    """
    def __init__(self, num_classes=4):
        super().__init__()

        self.mode = "multiclass"
        self.num_classes = num_classes
        init_channels = 32

        self.block1 = nn.Sequential(
            DepthwiseSeparableConv3d(1, init_channels, kernel_size=1, padding=1),
            nn.InstanceNorm3d(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))  # only pool H,W; depth=1
        )

        self.block2 = nn.Sequential(
            DepthwiseSeparableConv3d(init_channels, init_channels * 2, kernel_size=3, padding=1),
            nn.InstanceNorm3d(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        )

        self.block3 = nn.Sequential(
            DepthwiseSeparableConv3d(init_channels * 2, init_channels * 4, kernel_size=5, padding=1),
            nn.InstanceNorm3d(init_channels * 4),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        )

        self.block4 = nn.Sequential(
            DepthwiseSeparableConv3d(init_channels * 4, init_channels * 8, kernel_size=3, padding=1),
            nn.InstanceNorm3d(init_channels * 8),
            nn.LeakyReLU(inplace=True),
            nn.AdaptiveAvgPool3d((1, 1, 1))   # collapse D, H, W all to 1
        )

        self.classifier = nn.Sequential(
            nn.Linear(init_channels * 8, 128),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(128, self.num_classes)
        )

    def forward(self, **kwargs):
        x = kwargs["image"]
        labels = kwargs["label"]

        device = next(self.parameters()).device
        x = x.to(device)
        labels = labels.to(device)

        # (B, 1, H, W) -> (B, 1, 1, H, W): add a depth dimension of size 1
        x = x.unsqueeze(2)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = torch.flatten(x, 1)
        logits = self.classifier(x)

        loss = F.cross_entropy(logits, labels)
        y_prob = F.softmax(logits, dim=1)

        return {"loss": loss, "y_prob": y_prob, "y_true": labels}

In [10]:
from torch.utils.data import DataLoader

# Init model
model = PyHealthAlzheimersCNN_DepthwiseSep3D(
    num_classes=4
)

#Workers = 0 on MacOS, 4 on CoLab
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Set up PyHealth Trainer
trainer = Trainer(
    model=model,
    metrics=METRICS,
    device=DEVICE
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 1e-4, "weight_decay": 1e-5}
)

PyHealthAlzheimersCNN_DepthwiseSep3D(
  (block1): Sequential(
    (0): DepthwiseSeparableConv3d(
      (depthwise): Conv3d(1, 1, kernel_size=(1, 1, 1), stride=(1, 1, 1), padding=(1, 1, 1))
      (pointwise): Conv3d(1, 32, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    )
    (1): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): DepthwiseSeparableConv3d(
      (depthwise): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), groups=32)
      (pointwise): Conv3d(32, 64, kernel_size=(1, 1, 1), stride=(1, 1, 1))
    )
    (1): InstanceNorm3d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, cei

Epoch 0 / 50: 100%|██████████| 160/160 [02:54<00:00,  1.09s/it]

--- Train epoch-0, step-160 ---
loss: 1.1014



Evaluation: 100%|██████████| 40/40 [00:06<00:00,  5.98it/s]

--- Eval epoch-0, step-160 ---
accuracy: 0.4953
f1_macro: 0.1656
balanced_accuracy: 0.2500
loss: 1.0394



...

--- Eval epoch-48, step-7840 ---
accuracy: 0.5992
f1_macro: 0.3640
balanced_accuracy: 0.3626
loss: 0.8386




Epoch 49 / 50: 100%|██████████| 160/160 [02:53<00:00,  1.09s/it]

--- Train epoch-49, step-8000 ---
loss: 0.7803



Evaluation: 100%|██████████| 40/40 [00:06<00:00,  6.00it/s]

--- Eval epoch-49, step-8000 ---
accuracy: 0.5953
f1_macro: 0.3816
balanced_accuracy: 0.3793
loss: 0.8260


# Abilation Number 3

In [19]:
class PatchEmbedding(nn.Module):
    """Converts a CNN feature map into a sequence of patch tokens.

    Splits a (B, C, H, W) feature map into non-overlapping patches of size
    patch_size x patch_size, projects each flattened patch to embed_dim via
    a linear layer, prepends a learnable [CLS] token, and adds learnable
    positional embeddings.

    Args:
        in_channels: Number of channels in the input feature map.
        patch_size: Height and width of each square patch.
        embed_dim: Dimensionality of the projected patch tokens.
        feature_map_size: Spatial height (and width) of the input feature map.
            Assumed to be square.
    """

    def __init__(
        self,
        in_channels: int,
        patch_size: int,
        embed_dim: int,
        feature_map_size: int,
    ) -> None:
        super().__init__()
        self.patch_size = patch_size
        num_patches = (feature_map_size // patch_size) ** 2

        # Each patch is flattened to (C * patch_size * patch_size) dims
        patch_dim = in_channels * patch_size * patch_size
        self.projection = nn.Linear(patch_dim, embed_dim)

        # Learnable [CLS] token — one per batch item
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Learnable positional embeddings — one per patch + one for [CLS]
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Tokenize a CNN feature map into a patch sequence with CLS token.

        Args:
            x: Feature map of shape (B, C, H, W).

        Returns:
            Token sequence of shape (B, num_patches + 1, embed_dim).
        """
        B, C, H, W = x.shape
        p = self.patch_size

        # (B, C, H, W) → (B, C, H//p, W//p, p, p)
        x = x.unfold(2, p, p).unfold(3, p, p)
        # → (B, C, num_patches, p*p)
        x = x.contiguous().view(B, C, -1, p * p)
        # → (B, num_patches, C, p*p)
        x = x.permute(0, 2, 1, 3)
        # → (B, num_patches, C*p*p)
        x = x.contiguous().view(B, -1, C * p * p)

        # Project each patch to embed_dim
        x = self.projection(x)                         # (B, num_patches, embed_dim)

        # Prepend [CLS] token and add positional embeddings
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)          # (B, num_patches + 1, embed_dim)
        x = x + self.pos_embedding

        return x


class AlzheimerCNNViT(BaseModel):
    """CNN + Vision Transformer hybrid for Alzheimer's disease classification.

    Combines a two-block CNN backbone for local feature extraction with a
    Transformer encoder for global context modelling over patch tokens. The
    CNN reduces the 128x128 input to a 32x32 feature map, which is then split
    into 4x4 patches and fed to the Transformer. Classification is performed
    on the output [CLS] token.

    Architecture::

        Input (B, 1, 128, 128)
            │
        block1: Conv2d → InstanceNorm2d → LeakyReLU → MaxPool  →  (B, C, 64, 64)
        block2: Conv2d → InstanceNorm2d → LeakyReLU → MaxPool  →  (B, 2C, 32, 32)
            │
        PatchEmbedding (patch_size=4) → (B, 65, embed_dim)
            │
        TransformerEncoder (num_transformer_layers)
            │
        [CLS] token → Linear → LeakyReLU → Dropout → Linear → logits

    Paper:
        Liu et al. "On the design of convolutional neural networks for
        automatic detection of Alzheimer's disease." ML4H @ NeurIPS 2019.
        https://arxiv.org/abs/1911.03740

    Args:
        dataset: The dataset to train the model. Must expose input_schema,
            output_schema, and output_processors to satisfy BaseModel.
        init_channels: Base number of CNN filters. Blocks use C and 2C
            channels. Default is 32.
        embed_dim: Dimensionality of Transformer patch token embeddings.
            Default is 128.
        num_heads: Number of attention heads in the Transformer encoder.
            Must evenly divide embed_dim. Default is 4.
        num_transformer_layers: Number of Transformer encoder layers.
            Default is 4.
        classifier_hidden_dim: Number of units in the hidden FC layer of
            the classifier head. Default is 128.
        dropout: Dropout probability applied in the Transformer and before
            the output linear layer. Default is 0.1.

    Examples:
        >>> model = AlzheimerCNNViT(dataset=train_dataset)
        >>> ret = model(**data_batch)
        >>> print(ret.keys())
        dict_keys(['loss', 'y_prob', 'y_true', 'logit'])
    """

    def __init__(
        self,
        dataset,
        init_channels: int = 32,
        embed_dim: int = 128,
        num_heads: int = 4,
        num_transformer_layers: int = 4,
        classifier_hidden_dim: int = 128,
        dropout: float = 0.1,
        **kwargs,
    ) -> None:
        super(AlzheimerCNNViT, self).__init__(dataset=dataset)

        self.init_channels = init_channels
        self.embed_dim = embed_dim

        # ── CNN backbone: local feature extraction ────────────────────────
        # Two pooling stages reduce 128×128 → 32×32
        self.block1 = nn.Sequential(
            nn.Conv2d(1, init_channels, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),          # 128 → 64
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(init_channels, init_channels * 2, kernel_size=3, stride=1, padding=1),
            nn.InstanceNorm2d(init_channels * 2),
            nn.LeakyReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),          # 64 → 32
        )

        # ── Patch embedding: 32×32 feature map → 64 patch tokens + CLS ───
        # patch_size=4 gives (32/4)^2 = 64 patches
        self.patch_embed = PatchEmbedding(
            in_channels=init_channels * 2,
            patch_size=4,
            embed_dim=embed_dim,
            feature_map_size=32,
        )

        # ── Transformer encoder: global context over patch sequence ───────
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_transformer_layers,
        )

        # ── Classifier head: operates on the [CLS] token output ──────────
        output_size = self.get_output_size()
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, classifier_hidden_dim),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(classifier_hidden_dim, output_size),
        )

    def forward(self, **kwargs) -> Dict[str, torch.Tensor]:
        """Forward propagation.

        Args:
            **kwargs: Keys match the dataset's input_schema and output_schema.
                Expects an image tensor of shape (B, 1, H, W) and an integer
                label per sample.

        Returns:
            A dict with keys:
                - loss: scalar cross-entropy loss tensor.
                - y_prob: predicted probabilities, shape (B, num_classes).
                - y_true: ground-truth label tensor.
                - logit: raw logits, shape (B, num_classes).
        """
        # ── Extract inputs; self.device handles mps/cuda/cpu automatically ─
        x = kwargs[self.feature_keys[0]].to(self.device)
        labels = kwargs[self.label_keys[0]].to(self.device)

        # ── CNN local feature extraction ──────────────────────────────────
        x = self.block1(x)                     # (B, C,   64, 64)
        x = self.block2(x)                     # (B, 2C,  32, 32)

        # ── Patch tokenization ────────────────────────────────────────────
        x = self.patch_embed(x)                # (B, 65, embed_dim)

        # ── Transformer global context modelling ──────────────────────────
        x = self.transformer(x)                # (B, 65, embed_dim)

        # ── Extract [CLS] token (index 0) for classification ─────────────
        cls_output = x[:, 0, :]                # (B, embed_dim)

        # ── Classifier head ───────────────────────────────────────────────
        logits = self.classifier(cls_output)   # (B, num_classes)

        # ── Loss + metrics via BaseModel helpers ──────────────────────────
        loss = self.get_loss_function()(logits, labels)
        y_prob = self.prepare_y_prob(logits)

        return {
            "loss": loss,
            "y_prob": y_prob,
            "y_true": labels,
            "logit": logits,
        }

In [ ]:
from torch.utils.data import DataLoader

# Init model
model = AlzheimerCNNViT(
    dataset=train_samples,
    init_channels=32,
    classifier_hidden_dim=128,
    dropout=0.5,
)

#Num workers is 0 because I am running on MacOS, was previously 4
train_loader = get_dataloader(train_samples, batch_size=32, shuffle=True)
val_loader = get_dataloader(val_samples,   batch_size=32, shuffle=False)
test_loader = get_dataloader(test_samples,  batch_size=32, shuffle=False)

# Set up PyHealth Trainer
trainer = Trainer(
    model=model,
    metrics=METRICS,
    device=DEVICE
)

trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 1e-4, "weight_decay": 1e-5}
)

AlzheimerCNNViT(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
    (2): LeakyReLU(negative_slope=0.01, inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (patch_embed): PatchEmbedding(
    (projection): Linear(in_features=1024, out_features=128, bias=True)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_fe

Epoch 0 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-0, step-96 ---
loss: 1.1009


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 14.09it/s]

--- Eval epoch-0, step-96 ---
accuracy: 0.5098
f1_macro: 0.1688
balanced_accuracy: 0.2500
loss: 1.0514



...

Epoch 4 / 50:   0%|          | 0/96 [00:00<?, ?it/s]

--- Train epoch-4, step-480 ---
loss: 1.0573


Evaluation: 100%|██████████| 32/32 [00:02<00:00, 13.78it/s]

--- Eval epoch-4, step-480 ---
accuracy: 0.5098
f1_macro: 0.1688
balanced_accuracy: 0.2500
loss: 1.0317



Epoch 5 / 50:   0%|          | 0/96 [00:00<?, ?it/s]